# Test Scala Data Provenance

### Prerequisites

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

### Import libraries

In [2]:
val packageVersion = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % packageVersion)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

packageVersion: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension


import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension

### Initialize spark session

In [4]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .withExtensions(new ProvenanceExtension())
  // .config("spark.jars", s"../target/scala-2.13/dp-spark_2.13-$packageVersion.jar")
  // .config("spark.sql.extensions", "org.dataprov.dp.ProvenanceExtension")
  .config("spark.provenance.enabled", "true")
  .getOrCreate()


println(s"Spark provenance enabled: ${spark.conf.get("spark.provenance.enabled")}")
  

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/13 16:29:24 INFO SparkContext: Running Spark version 4.1.1
26/05/13 16:29:24 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/05/13 16:29:24 INFO SparkContext: Java version 17.0.10+7
26/05/13 16:29:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/13 16:29:24 INFO ResourceUtils: ==============================================================
26/05/13 16:29:24 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/13 16:29:24 INFO ResourceUtils: ==============================================================
26/05/13 16:29:24 INFO SparkContext: Submitted application: notebook
26/05/13 16:29:24 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/05/13 16:29:24 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/05/13 16:29:24 INFO SecurityManager: Changing view acls groups to

Spark provenance enabled: true


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@81fa4af

### Create spark dataframe

We create a dataframe with duplicates to test the provenance annotation.
 
The provenance annotation will count the number of duplicates for each tuple.

In [5]:
// val df: DataFrame = spark.createDataFrame(
//     Seq(
//         ("a", "b", "c"),
//         ("a", "b", "c"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("d", "b", "e"),
//         ("f", "g", "e")
//     )
// ).toDF("A", "B", "C")

// df.show()

val df: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+



df: DataFrame = [A: string, B: string ... 1 more field]

In [6]:
val dfWithProvenance : DataFrame = df.addProvenanceColumn
dfWithProvenance.printSchema()

dfWithProvenance.show()

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: long (nullable = false)

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|              0|
|  d|  b|  e|              1|
|  f|  g|  e|              2|
+---+---+---+---------------+



dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [7]:
def getProvAttr(plan: LogicalPlan, provenanceColName: String): Attribute =
        if (LogicalPlanIntegrity.canGetOutputAttrs(plan)) plan.output.find(_.name == provenanceColName).get else throw new IllegalArgumentException("Plan is not resolved")

getProvAttr(dfWithProvenance.queryExecution.analyzed, "_provenance_tag")

defined function getProvAttr
res7_1: Attribute = AttributeReference(
  name = "_provenance_tag",
  dataType = LongType,
  nullable = false,
  metadata = {}
)

In [8]:
val df2 : DataFrame = df
    .select("A", "B")
    .join(df.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2.show()

val df3: DataFrame = df
  .select("A", "C")
  .join(df.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")

val df2WithProvenance : DataFrame = dfWithProvenance
    .select("A", "B")
    .join(dfWithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show()

val df3WithProvenance: DataFrame = dfWithProvenance
  .select("A", "C")
  .join(dfWithProvenance.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")

df3WithProvenance.show()


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|        (0 ⊗ 0)|
|  a|  b|  e|        (0 ⊗ 1)|
|  d|  b|  c|        (1 ⊗ 0)|
|  d|  b|  e|        (1 ⊗ 1)|
|  f|  g|  e|        (2 ⊗ 2)|
+---+---+---+---------------+

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|        (0 ⊗ 0)|
|  d|  b|  e|        (1 ⊗ 1)|
|  d|  g|  e|        (1 ⊗ 2)|
|  f|  b|  e|        (2 ⊗ 1)|
|  f|  g|  e|        (2 ⊗ 2)|
+---+---+---+---------------+



df2: DataFrame = [A: string, B: string ... 1 more field]
df3: DataFrame = [A: string, B: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [9]:
val df4 = df2.union(df3).distinct().orderBy("A", "B", "C")
df4.show()

val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+---------------+
|  A|  B|  C|_provenance_tag|
+---+---+---+---------------+
|  a|  b|  c|        (0 ⊗ 0)|
|  a|  b|  e|        (0 ⊗ 1)|
|  d|  b|  c|        (1 ⊗ 0)|
|  d|  b|  e|        (1 ⊗ 1)|
|  d|  g|  e|        (1 ⊗ 2)|
|  f|  b|  e|        (2 ⊗ 1)|
|  f|  g|  e|        (2 ⊗ 2)|
+---+---+---+---------------+



df4: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 1 more field]
df4WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

In [10]:
val df5 = df4.select("A", "C").distinct().orderBy("A", "C")
df5.show()

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show()

+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+---------------+
|  A|  C|_provenance_tag|
+---+---+---------------+
|  a|  c|        (0 ⊗ 0)|
|  a|  e|        (0 ⊗ 1)|
|  d|  c|        (1 ⊗ 0)|
|  d|  e|        (1 ⊗ 1)|
|  d|  e|        (1 ⊗ 2)|
|  f|  e|        (2 ⊗ 1)|
|  f|  e|        (2 ⊗ 2)|
+---+---+---------------+



df5: Dataset[org.apache.spark.sql.Row] = [A: string, C: string]
df5WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, C: string ... 1 more field]